# BirdCLEF 2026 - Exploratory Data Analysis

This notebook explores the competition dataset: audio samples, species distribution, and spectrogram visualizations.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import IPython.display as ipd

DATA_DIR = "../data/birdclef-2026"
SAMPLE_RATE = 32000

## 1. Load Metadata

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train_metadata.csv"))
print(f"Training samples: {len(train_df)}")
print(f"Number of species: {train_df['primary_label'].nunique()}")
print(f"\nColumns: {list(train_df.columns)}")
train_df.head()

## 2. Species Distribution

In [ ]:
species_counts = train_df['primary_label'].value_counts()
print(f"Most common species: {species_counts.head(10)}")
print(f"\nLeast common species: {species_counts.tail(10)}")
print(f"\nMean samples per species: {species_counts.mean():.1f}")
print(f"Median samples per species: {species_counts.median():.1f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
species_counts.plot(kind='bar', ax=axes[0], title='Samples per Species')
axes[0].set_xlabel('Species')
axes[0].set_ylabel('Count')
axes[0].tick_params(labelbottom=False)

species_counts.plot(kind='hist', bins=50, ax=axes[1], title='Distribution of Sample Counts')
axes[1].set_xlabel('Number of Samples')
axes[1].set_ylabel('Number of Species')
plt.tight_layout()
plt.show()

## 3. Audio Exploration

In [ ]:
# Load and visualize a sample audio file
sample = train_df.iloc[0]
audio_path = os.path.join(DATA_DIR, "train_audio", sample["filename"])
print(f"Species: {sample['primary_label']}")
print(f"File: {sample['filename']}")

y, sr = librosa.load(audio_path, sr=SAMPLE_RATE)
print(f"Duration: {len(y)/sr:.2f}s | Sample rate: {sr}Hz")

# Play audio
ipd.Audio(y, rate=sr)

In [ ]:
# Waveform and spectrogram
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Waveform
librosa.display.waveshow(y, sr=sr, ax=axes[0])
axes[0].set_title(f'Waveform - {sample["primary_label"]}')

# Mel spectrogram
mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmin=20, fmax=16000)
mel_db = librosa.power_to_db(mel_spec, ref=np.max)
librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel', ax=axes[1], fmin=20, fmax=16000)
axes[1].set_title('Mel Spectrogram')
plt.colorbar(axes[1].images[0], ax=axes[1], format='%+2.0f dB')

# STFT spectrogram
D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='log', ax=axes[2])
axes[2].set_title('STFT Spectrogram')
plt.colorbar(axes[2].images[0], ax=axes[2], format='%+2.0f dB')

plt.tight_layout()
plt.show()

## 4. Compare Multiple Species

In [ ]:
# Visualize mel spectrograms for different species
species_samples = train_df.groupby('primary_label').first().head(6).reset_index()

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for idx, (_, row) in enumerate(species_samples.iterrows()):
    ax = axes[idx // 3, idx % 3]
    audio_path = os.path.join(DATA_DIR, "train_audio", row["filename"])
    y, sr = librosa.load(audio_path, sr=SAMPLE_RATE, duration=5)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmin=20, fmax=16000)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel', ax=ax, fmin=20, fmax=16000)
    ax.set_title(row['primary_label'], fontsize=10)

plt.tight_layout()
plt.show()

## 5. Audio Duration Analysis

In [ ]:
# Analyze audio durations (from metadata if available, otherwise sample)
if 'duration' in train_df.columns:
    durations = train_df['duration']
else:
    # Sample 200 files to estimate duration distribution
    sample_files = train_df.sample(min(200, len(train_df)), random_state=42)
    durations = []
    for _, row in sample_files.iterrows():
        path = os.path.join(DATA_DIR, "train_audio", row["filename"])
        try:
            dur = librosa.get_duration(path=path)
            durations.append(dur)
        except Exception:
            continue
    durations = pd.Series(durations)

print(f"Duration stats (seconds):")
print(durations.describe())

plt.figure(figsize=(10, 4))
plt.hist(durations, bins=50, edgecolor='black')
plt.xlabel('Duration (seconds)')
plt.ylabel('Count')
plt.title('Audio Duration Distribution')
plt.tight_layout()
plt.show()